# Example Notebook: Exploring Unlabeled Text

This notebook gives a small, runnable example for exploring text when you do not yet have labels.

The main ideas are:

- inspect the texts directly;
- group similar texts with TF-IDF and clustering;
- retrieve texts related to a concept using embeddings;
- move from whole-document retrieval to chunk-level retrieval for narrower concepts.

In this example, labels are still available only so the exploratory methods can be checked against a known reference. In a real unlabeled workflow, you would rely on inspection and interpretation of the retrieved or clustered texts.

In [1]:
import importlib.util
import re
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

if importlib.util.find_spec('sentence_transformers') is None:
    print('Installing sentence-transformers...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers'], check=True)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/opt/anaconda3/envs/aaib26/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load a small real text dataset

We use a two-class subset of `20 Newsgroups` so the notebook stays quick to run while still containing realistic vocabulary and longer passages. The same workflow can later be reused for other text collections when labels are not yet available.

In [2]:
categories = ['rec.autos', 'sci.med']
label_map = {'rec.autos': 'autos', 'sci.med': 'med'}

dataset = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
)

df = pd.DataFrame({
    'text': dataset.data,
    'label': [label_map[dataset.target_names[i]] for i in dataset.target],
})

df['word_count'] = df['text'].str.split().str.len()
df = df[df['word_count'] >= 20].copy()

sampled_groups = []
for _, group in df.groupby('label'):
    sampled_groups.append(group.sample(n=min(len(group), 220), random_state=42))

df = pd.concat(sampled_groups, ignore_index=True)

display(df.head())
print('Rows:', len(df))

,text,label,word_count
0,My son is considering the purchase of a 71 MGB...,autos,275
1,\nLimited Tort Option will lower your rates. I...,autos,56
2,\nI think you mean ARPA; AARP is the American ...,autos,238
3,I'm about to buy a new car and finance some of...,autos,171
4,I'm having an interesting problem with my girl...,autos,214


Rows: 440


In [3]:
for label_name in ['autos', 'med']:
    example = df.loc[df['label'] == label_name, 'text'].sample(n=1).iloc[0]
    print(f'Random example from {label_name}:')
    print(example[:500])
    print()

Random example from autos:
(I deleted your name because I don't want to sound accusative in my remark)
I'm not going to argue the issue of carrying weapons, but I would ask you if 
you would have thought seriously about shooting a kid for setting off your
alarm?  I can think of worse things in the world.  Glad you got out of there
before they did anything to give you a reason to fire your gun.

We can all ask "what's happening to society these days", but don't forget to
ask another important question too: What effort am I

Random example from med:
I am writing this to find out the following:

1.)	Any information on surgery to prevent reflux esophagitis.

2.)	The name(s) of a doctor(s) who specialize in such surgery.

3.)	Information on reflux esophagitis which leads to cancer.

My boyfriend, age 34 and otherwise in good health, was diagnosed with 
reflux esophagitis and a hiatal hernia about 2 years ago.  At that time he 
saw a gastroenterologist and has tried acid controllers (Mylanta

## 2. Group texts into themes or reasons

A traditional NLP route for unlabeled text is to convert documents into TF-IDF features, cluster them, inspect the top terms, and then read representative examples from each cluster.

The clusters are only a starting point. You still need to inspect the texts and decide whether the groups are meaningful.

In [4]:
unsup_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words='english', max_features=1000)
X_unsup = unsup_vectorizer.fit_transform(df['text'])

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X_unsup)

cluster_df = df[['text', 'label']].copy()
cluster_df['cluster'] = cluster_ids

cluster_counts = cluster_df['cluster'].value_counts().sort_index().rename('count')
print('Documents per cluster:')
display(cluster_counts.to_frame())

terms = unsup_vectorizer.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort(axis=1)[:, ::-1]

for cluster_id in range(n_clusters):
    top_terms = [terms[index] for index in order_centroids[cluster_id, :8]]
    print(f'Cluster {cluster_id}:')
    print('Top terms:', ', '.join(top_terms))

    cluster_rows = cluster_df[cluster_df['cluster'] == cluster_id].copy()
    cluster_matrix = X_unsup[cluster_rows.index]
    distances = kmeans.transform(cluster_matrix)[:, cluster_id]
    representative_rows = cluster_rows.assign(distance=distances).nsmallest(3, 'distance')

    print('Example texts:')
    for text in representative_rows['text']:
        print('-', text[:180].replace('\n', ' '))
    print()

Documents per cluster:


,count
cluster,
0,92
1,19
2,205
3,124


Cluster 0:
Top terms: thanks, know, like, new, mail, does, lot, computer
Example texts:
- Hello, 	I am looking to slightly increase the performance of my 89 Honda Civic Si.  I was wondering if anyone could suggest upgrades that were not too drastic.  I thought that one 
- I'm thinking of buying a new Dodge Intrepid - Has anyone had any experiences that they'd like to share?  Thanks.
- I have a 1982 Regal and I am interested in buying a fiberglass hood, trunk, and bumpers for it.  Does anybody know of a company who makes fiberglass parts for Regals ??   		Thanks 

Cluster 1:
Top terms: chastity, geb, intellect geb, dsl, dsl pitt, edu shameful, n3jxp skepticism, n3jxp
Example texts:
-  Senile keratoses.  Have nothing to do with the liver.   --  ---------------------------------------------------------------------------- Gordon Banks  N3JXP      | "Skepticism is 
-   Gosh, Jesse is that famous now?  He was my intern.  Landau not liking it makes me like it out of spite.  (Just kidding, Bil

## 3. Semantic search with embeddings

If you have a concept in mind rather than a label, embeddings from pretrained models let you search by meaning rather than by exact keyword overlap.

This works best for fairly concrete semantic similarity. It is better treated as a retrieval baseline than as a reliable detector of complex constructs.

In [5]:
# Load a pretrained embedding model for semantic retrieval.
embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')
# Turn every full document into a numeric embedding vector.
document_embeddings = embedding_model.encode(df['text'].tolist(), show_progress_bar=False)

query = 'car engine problem, repair costs, mechanic, vehicle fault'
expected_label = 'autos'
# Embed the query in the same vector space as the documents.
query_embedding = embedding_model.encode([query], show_progress_bar=False)

# Higher cosine similarity means the document is closer to the query in meaning.
similarities = cosine_similarity(query_embedding, document_embeddings)[0]

semantic_hits = df[['text', 'label']].copy()
semantic_hits['similarity'] = similarities
semantic_hits = semantic_hits.sort_values('similarity', ascending=False).reset_index(drop=True)

label_similarity_summary = (
    semantic_hits.groupby('label')['similarity']
    .agg(['mean', 'median', 'max'])
    .round(3)
    .sort_values('mean', ascending=False)
)

top_k = 20
top_k_hits = semantic_hits.head(top_k).copy()
top_k_label_counts = top_k_hits['label'].value_counts().rename('count').to_frame()
top_k_label_counts['proportion'] = (top_k_label_counts['count'] / top_k).round(3)

# Because this demo still has labels, we can check whether retrieval is pointing to the expected class.
top_hit_rate = (top_k_hits['label'] == expected_label).mean()

# Show a readable preview instead of letting pandas truncate unpredictably.
semantic_preview = semantic_hits.head(8)[['label', 'similarity', 'text']].copy()
semantic_preview['text'] = semantic_preview['text'].str.replace('\n', ' ', regex=False).str.slice(0, 200)

print('Query:', query)
print(f'Expected label for this query: {expected_label!r}')
print()
print('Similarity summary by label:')
display(label_similarity_summary)
print(f'Label counts in the top {top_k} hits:')
display(top_k_label_counts)
print(f"Proportion of top {top_k} hits with label {expected_label!r}: {top_hit_rate:.3f}")
print()
print('Top semantic matches (first 200 characters):')
with pd.option_context('display.max_colwidth', None):
    display(semantic_preview)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7694.76it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: car engine problem, repair costs, mechanic, vehicle fault
Expected label for this query: 'autos'

Similarity summary by label:


,mean,median,max
label,,,
autos,0.562,0.565,0.693
med,0.448,0.451,0.572


Label counts in the top 20 hits:


,count,proportion
label,,
autos,20,1.0


Proportion of top 20 hits with label 'autos': 1.000

Top semantic matches (first 200 characters):


,label,similarity,text
0,autos,0.693254,"My BBB Autoline arbitration experience is over. The outcome was decidedly mixed. I won the battle but lost the war. The arbitrator found that the car was defective, but decided to offer a repurc"
1,autos,0.689895,"I just ordered a Saturn SL1 after considering a few imports. Frankly, the Saturn way of doing business and service was a *very* big plus. I hadn't bought a new car since I bought my Honda 4WD back in"
2,autos,0.677971,did you account for depreciation? i seriously doubt that a taurus would rack up an extra $4000 in repair costs over 5 years.
3,autos,0.676861,"I looked at that Bimmer yesterday. It's an '81, has about 90kmi, according to owner (odometer stopped working at 68Kmi). Drivess well, sounds good, body is OK, he wants $3000. i liked the car, despite"
4,autos,0.673533,"I was in the great storm.....my Mazda MPV was damaged so bad they are going to replace the top, doors and hood. It is Black so they will repaint the entire vehicle...estimated cost around $7000 and"
5,autos,0.670582,"Hello everyone, I have an insurance question. Allstate insurance SITUATION: Person wrecks car. Car is drivable to dealer. Person reports accident (no other cars involved). Driver estimates damag"
6,autos,0.670426,@>> @>>Has anyone had any experience with GEICO's extended @>>warranty plan. It seems to be slightly less expensive than @>>the normal dealer-sponsored policy. @>> @>and once again....*never* buy ext
7,autos,0.666587,"I've found mine ('93 Probe GT) to do quite well. [window problem deleted, artical has been trimmed] I've not had any of the air or leakage problems that have been reported but do get the squeal"


## 4. Chunk-level semantic search for annotation

Whole-document embeddings can be too coarse when each text contains several different ideas. For annotation tasks, it is often better to split the text into smaller units and search those units instead.

A single query string often behaves too much like soft keyword matching. For narrower concepts, a better starting point is to describe the concept with several short example statements and average their embeddings into one concept prototype.

In [6]:
def sentence_chunks(text, min_words=8):
    pieces = re.split(r'(?<=[.!?])\s+', str(text).replace('\n', ' '))
    return [piece.strip() for piece in pieces if len(piece.split()) >= min_words]

chunk_rows = []
for doc_id, row in df[['text', 'label']].iterrows():
    for chunk_id, chunk_text in enumerate(sentence_chunks(row['text'])):
        chunk_rows.append({
            'doc_id': doc_id,
            'chunk_id': chunk_id,
            'label': row['label'],
            'chunk_text': chunk_text,
        })

chunk_df = pd.DataFrame(chunk_rows)
print('Total chunks:', len(chunk_df))

concept_name = 'medication side effects and adverse reactions'
concept_examples = [
    'the patient reported side effects after starting the medication',
    'the treatment caused nausea, dizziness, and other adverse reactions',
    'the doctor discussed whether the symptoms were caused by medication side effects',
    'the medicine seemed effective but produced unpleasant side effects',
]
expected_chunk_label = 'med'

# Embed each chunk so we can rank smaller passages instead of whole documents.
chunk_embeddings = embedding_model.encode(chunk_df['chunk_text'].tolist(), show_progress_bar=False)
# Embed several example statements and average them into one prototype vector.
concept_embeddings = embedding_model.encode(concept_examples, show_progress_bar=False)
concept_embedding = concept_embeddings.mean(axis=0, keepdims=True)

# Score each chunk against the prototype concept.
chunk_similarities = cosine_similarity(concept_embedding, chunk_embeddings)[0]

chunk_hits = chunk_df.copy()
chunk_hits['similarity'] = chunk_similarities
chunk_hits = chunk_hits.sort_values('similarity', ascending=False).reset_index(drop=True)

chunk_label_summary = (
    chunk_hits.groupby('label')['similarity']
    .agg(['mean', 'median', 'max'])
    .round(3)
    .sort_values('mean', ascending=False)
)

chunk_top_k = 20
chunk_top_hits = chunk_hits.head(chunk_top_k).copy()
chunk_top_label_counts = chunk_top_hits['label'].value_counts().rename('count').to_frame()
chunk_top_label_counts['proportion'] = (chunk_top_label_counts['count'] / chunk_top_k).round(3)

# Again, labels are used here only as a sanity check for the teaching example.
chunk_hit_rate = (chunk_top_hits['label'] == expected_chunk_label).mean()
matched_docs = chunk_top_hits['doc_id'].nunique()

print('Concept:', concept_name)
print('Prototype examples used to define the concept:')
for example in concept_examples:
    print('-', example)
print()
print(f'Expected label for this concept: {expected_chunk_label!r}')
print()
print('Chunk similarity summary by label:')
display(chunk_label_summary)
print(f'Label counts in the top {chunk_top_k} chunk hits:')
display(chunk_top_label_counts)
print(f"Proportion of top {chunk_top_k} chunk hits with label {expected_chunk_label!r}: {chunk_hit_rate:.3f}")
print(f'Distinct source documents represented in the top {chunk_top_k} chunk hits: {matched_docs}')
print()
print('Top chunk matches:')
chunk_display = chunk_top_hits[['label', 'similarity', 'doc_id', 'chunk_text']].copy()

with pd.option_context('display.max_colwidth', None):
    display(
        chunk_display.style.set_properties(
            subset=['chunk_text'],
            **{'white-space': 'pre-wrap', 'text-align': 'left'}
        )
    )

Total chunks: 3473
Concept: medication side effects and adverse reactions
Prototype examples used to define the concept:
- the patient reported side effects after starting the medication
- the treatment caused nausea, dizziness, and other adverse reactions
- the doctor discussed whether the symptoms were caused by medication side effects
- the medicine seemed effective but produced unpleasant side effects

Expected label for this concept: 'med'

Chunk similarity summary by label:


,mean,median,max
label,,,
med,0.521,0.517,0.746
autos,0.444,0.443,0.617


Label counts in the top 20 chunk hits:


,count,proportion
label,,
med,20,1.0


Proportion of top 20 chunk hits with label 'med': 1.000
Distinct source documents represented in the top 20 chunk hits: 14

Top chunk matches:


,label,similarity,doc_id,chunk_text
0,med,0.746181,434,Does short-term use of the drug ever produce long-term side-effects after the use of the drug?
1,med,0.739790,263,It was also to discuss real-life experiences and side effects of the above mentioned.
2,med,0.731048,351,"I explained to her that I was following up at the doctor's request, and that I was worried because the pain episodes were becoming more frequent and the medication did not seem effective."
3,med,0.730777,410,You could have real effects *and* placebo effects; people may have allergies in addition.
4,med,0.730667,411,"(kinda like mixing alcohol and antihistamines!) Unfortunately, doctors prescribe drugs to treat the side effects of the drugs a patient is receiving."
5,med,0.718016,411,"And because some drugs potentiate the effect of each other, they can make the side effects all the worse, and even dangerous."
6,med,0.705379,437,Diet affects this mix as does the use of antibiotics.
7,med,0.704006,302,Taking Premarin can certainly cause migraines in some women.
8,med,0.701888,289,"Maybe those doctors who are reading this who have a practice and are confronted by a patient having symptoms that could be due to the ""hypothetical yeast overgrowth"" (e.g., they fit some of the profiles the pro-yeast people have identified), should consider anti-fungal therapy IF all other avenues have been exhausted."
9,med,0.701631,289,"I was concerned, too, because of the toxicity of vitamin A."


## 5. Takeaways

- Start by reading examples before relying on any unsupervised output.
- TF-IDF clustering is a useful first pass for surfacing broad groups.
- Embedding retrieval is useful for concept search, but it is still a heuristic.
- Chunk-level retrieval is often more useful than whole-document retrieval when you want to annotate passages or count concept mentions.
- Human review is still needed to decide whether clusters or retrieved passages are actually meaningful.
- The same workflow can be reused on other text collections when labels are not yet available.